In [1]:
import pandas as pd
import re
import numpy as np
import sys

# Functions for computing conditional entropy

The code will depend on the distinctions between types of segments not made explicitly in `ParaCzech`. The two dictionaries `orth_segments` and `phon_segments` provide the relevant information.

In addition those dictionaries provide useful ways to name segments involved in stem alternations in a parallel manner whether orthographic or phonologica transcriptions are used, using the orthographic transcription as a key.

In [2]:

orth_segments = {'hard': ['ch', 'h', 'k', 'r', 'd', 't', 'n', 'g'],
            'soft': ['ž', 'š', 'č', 'ř', 'c', 'j', 'ď', 'ť', 'ň'],
            'ambiguous': ['b', 'f', 'l', 'm', 'p', 's', 'v', 'z'],
            'vowel': ['a','á','e','é','ě','i','í','o','ó','u','ú','ů','y','ý'],
            'ů':'ů',
            'e':'e',
            'o':'o'}

orth_segments['consonants'] = orth_segments['hard']+orth_segments['soft'] + orth_segments['ambiguous']

phon_segments = {'hard': ['x', 'ɣ', 'ɦ', 'k', 'r', 'd', 't', 'n', 'ɡ', 'ŋ'],
            'soft': ['d͡ʒ','d͡z', 'ʒ', 'ʃ', 't͡ʃ', 'r̝', 'r̝̊', 't͡s', 'j', 'ɟ', 'c', 'ɲ'],
            'ambiguous': ['b', 'f', 'l', 'm','ɱ','p', 's', 'v', 'z'],
            'vowel': ['ɛː', 'ɔ', 'ɔu̯', 'ɛ', 'iː', 'u', 'a', 'ɪ', 'ɔː', 'au̯', 'aː', 'uː', 'ɛu̯'],
            'ů':'uː',
            'e':'ɛ',
            'o':'ɔ'}


phon_segments['consonants'] = phon_segments['hard']+phon_segments['soft'] + phon_segments['ambiguous']

def seg_type(s,segments):
    '''Associates a special symbol with segment classes'''
    if s in segments['hard']:
        return 'H'
    if s in segments['soft']:
        return 'S'
    if s in segments['ambiguous']:
        return 'A'
    if s in segments['vowel']:
        return 'V'




The first function of interest, `alt`, looks at pairs of forms and returns a description of the alternation between these two forms.

The function does two nonobvious things hardcoding specific knowledge of Czech grammar:
- Description of alternations crucially classifies the last common segment between the two sequences as a vowel (V), or a hard (H), soft (S), or ambiguous (A) consonant. This amounts to a very restricted form of Minimal Generalization (Albright 2002), where a 4-valued feature is the only thing than can be generalized, and is always generalized.
- The funcion specifically looks for alternations between _ů_ (resp. _uː_ when using phonemic transcriptions) an

In [17]:
def alt(x,y,segments):
    '''takes as input a pair of strings and returns the pattern relating these two strings.
    The function assumes that the strings are space-separated lists of graphemes or phonemes
    This version implements
        - A classification of stem final segments as vowels, or hard, soft, or ambiguous consonants
        - Identification of stem internal changes of the form eC#~CX and ůC#~oCX
    The way this is done is very inelegant, but seems to work.'''
    # First we identify a common initial substring using a regular expression.
    m = re.match(r'^(.+)(| .+)~\1(| .+)$',x+'~'+y)
    if m:
        left = m.group(2).strip().split(' ') if len(m.group(2))>0 else []
        right = m.group(3).strip().split(' ')  if len(m.group(3))>0 else []
        stem = m.group(1).strip().split(' ')
        # Then we go through possible prefinal alternations. 
        # Case by case:
        # (i) we identify the final nonalternating segment, replace it with its class label, and store the result in cons
        # (ii) we place that correctly with respect to the other pieces of the alternation
        #
        # First case: type "o t e c~o t c u"
        if len(left)==2 and len(right)>=1 and left[1] in segments['consonants'] and left[0] == segments['e'] and right[0] == left[1]:
            cons = seg_type(right[0],segments)
            left_new = [left[0],cons]
            right_new = [cons]+right[1:]
        # Second case: type "o t c e~o t e c"
        elif len(right)==2 and len(left)>=1 and right[1] in segments['consonants']  and right[0] == segments['e'] and left[0] == right[1]:
            cons = seg_type(left[0],segments)
            left_new= [cons]+left[1:]
            right_new = [right[0],cons]
        # Third case: type "d ů m ~ d o m u"
        elif len(left)==2 and len(right)>=2 and left[1] in segments['consonants'] and left[0] == segments['ů'] and right[0]== segments['o'] and right[1] == left[1]:
            cons = seg_type(right[1],segments)
            left_new = [left[0],cons]
            right_new = [right[0],cons]+right[2:]
        # Fourth case: type "d o m u ~ d ů m"
        elif len(left)>=2 and len(right)==2 and right[1] in segments['consonants'] and right[0] == segments['ů'] and left[0]== segments['o'] and left[1] == right[1]:
            cons = seg_type(right[1],segments)
            left_new = [left[0],cons]+left[2:]
            right_new = [right[0],cons]
        # Fifth case: if no alternation was identified, we just record the type of the last stem segment, if there is one
        elif len(stem)>0:
            cons = seg_type(stem[-1],segments)
            left_new = [cons]+left
            right_new = [cons]+right
        return ' '.join(left_new+['~']+ right_new)
    # The only remaining situation is where there is no common initial substring. This may correspond to suppletion or to some alternation 
    # in the very first segment. In that case we just record the alternation as a change between the two full strings.
    return x+' ~ '+y

Uncomment this for examples showing that `alt` produces results that cohere with what is usually said about Czech.

In [4]:
# print("Orthographic forms")
# for first,second in [("h r a d","h r a d u"),("t á t a","t á t y"), ("o t e c","o t c e"), ("d ů m","d o m u")]:
#     print(first.replace(" ",""), second.replace(" ",""),alt(first,second,orth_segments))
# print()
# 
# print("Phonological forms")
# for first,second in [("ɦ r a t","ɦ r a d u"),("t aː t a","t aː t ɪ"), ("ʔ ɔ t ɛ t͡s","ʔ ɔ t t͡s ɛ"), ("d uː m","d ɔ m u")]:
#     print(first.replace(" ",""), second.replace(" ",""),alt(first,second,phon_segments))
#   

The second function of interest, `applicable`, records whether a form matches the left hand side of an alternation. This amounts to aligning the string and the left hand side of the pattern on the right, and then check position by position that:
- If the pattern has one of the special characters S, H, A or V in this position, the string has a segment of the corresponding clas in the same position.
- Otherwise the string and pattern have the same grapheme or phoneme in that position.

In [5]:

def applicable(pattern,s,segments):
    '''Given a pattern and a string, returns True if that pattern applies to that string.
        The function assumes that the strings are space-separated lists of graphemes or phonemes
    '''
    pattern_segs = pattern.split('~')[0].rstrip().split(' ')[::-1]
    string_segs = s.split(' ')[::-1]
    if len(string_segs)<len(pattern_segs):
        return False
    for i in range(len(pattern_segs)):
        if pattern_segs[i] == 'S':
            if string_segs[i] not in segments['soft']:
                return False
        elif pattern_segs[i] == 'H':
            if string_segs[i] not in segments['hard']:
                return False
        elif pattern_segs[i] == 'A':
            if string_segs[i] not in segments['ambiguous']:
                return False
        elif pattern_segs[i] == 'V':
            if string_segs[i] not in segments['vowel']:
                return False
        elif pattern_segs[i]!=string_segs[i]:
            return False
    return True

#applicable('ů H~o H y','b ů h',orth_segments)


The function `entropy`just takes a pandas series representing a probability distribution and computes the corresponding entropy value.

In [6]:
def entropy(S):
    """Given a probability distribution in the form of a pandas Series,
    returns the entropy of that distribution."""
    S = S[S!=0] 
    return abs( - S.apply(lambda x: x * np.log2(x)).sum() )


The function `cond_ent` takes as input a dataframe with the  structure of a paralex forms table, as well of the labels of two cells documented in that dataframe, and information as to whether phonological or orthpographic forms are to be used. It then proceeds as follows:
1. Creates a new dataframe `pairs` with one lexeme per row and the cells of interest in columns.
2. Drops lexemes with either overabundance or no data for one of the cells
3. Computes alternations using `alt`
4. Computes for each form in cell1 the set of applicable patterns using `applicable`
5. Groups lexemes by sets of lexemes that have the same applicable patterns
6. Computes the entropy of the type frequency distribution of patterns within each groups
7. Computes the conditional entropy as the average of the entropy within each group, weighted by the relative type frequency of the groups.

If `debug=True` is passed, the function outputs tables exemplifying the patterns within each class and reports entropy local to that class.

In [7]:
def cond_ent(cell1,cell2,df,form_type,debug=False):
    """Takes two cells, a dataframe and a form type (phon or orth) and
    returns the conditional entropy of the second cell given the first cell, à la Bonami & Boyé 2014.
    The output is a dictionary with both the result of the entropy computation and the size of the output.
    If debug is true, prints all intermediate computations.
    """
    if debug:
        print()
        print('='*30)
        print(cell1,'→',cell2)
        print('='*30)
        print()
    # First we decide which data to use based on form_type
    segments = eval(form_type+'_segments')
    values = form_type+'_form'
    # We extract pairs of forms in the two relevant cells, excluding missing forms and overabundant cells 
    sel = df[((df.cell==cell1)|(df.cell==cell2))]
    pairs = sel.pivot_table(columns="cell",index="lexeme",values=values,aggfunc = lambda l:';'.join(l))
    pairs = pairs[[cell1,cell2]].dropna()
    pairs = pairs[(~pairs[cell1].str.contains(';'))& (~pairs[cell2].str.contains(';'))]

 
    pairs['alt'] = pairs.apply(lambda S: alt(S[cell1],S[cell2],segments),axis=1)
    
    size = len(pairs)
    pairs = pairs[pairs.alt.notna()]
    if debug:
        if  len(pairs)!=size:
            print('Dropped', size-len(pairs), 'pairs with no identified alternation')
        print(len(pairs), 'pairs of forms examined')
        
    patterns = set(pairs.alt)
    if debug:
        print (len(patterns), 'patterns')

    # Next we find out, for each predictor form, which patterns it is compatible with 
    # (ie which patterns could conceivably apply to it)
    pairs['applicable'] = pairs[cell1].apply(lambda s: frozenset({p for p in patterns if applicable(p,s,segments)}))

    # We are going to compute, for each of lexemes to which the same patterns are applicable, 
    # the entropy of the distribution of those patterns. We will store the results in local_ents. 
    local_ents = {}
    if debug:
        # It is convenient to have an extra column for purposes of displaying the results
        pairs['count']=1
        # This will hold what is to be displayed
        to_be_visualized = []
    # Now we actually compute the entropies    
    for local_patterns,group in pairs.groupby('applicable'):
        local_ents[local_patterns] = entropy(group.alt.value_counts(normalize=True))
        if debug:
            ending = '; '.join({p.split('~')[0] for p in local_patterns})
            table = group.groupby('alt').agg({'count':'sum',cell1:lambda S:S.iloc[0], cell2:lambda S:S.iloc[0]})
            table.index = [pattern.replace(' ','') for pattern in table.index] 
            local_patterns_viz = {pattern.replace(' ','') for pattern in local_patterns}
            to_be_visualized.append((local_patterns_viz,ending.replace(' ',''),table,table['count'].sum(),local_ents[local_patterns]))
    
    if debug:
        print(len(to_be_visualized), 'classes of predictor shapes\n')
        to_be_visualized = pd.DataFrame(to_be_visualized,columns=['patterns','ending','table','size','entropy'])
        to_be_visualized = to_be_visualized.sort_values('size',ascending=False).reset_index()
        for i in to_be_visualized.index:                  
            print('Class',i+1,':',to_be_visualized.at[i,'size'], 'lexemes ending in', to_be_visualized.at[i,'ending'])
            print('Patterns: ', end= '')
            print(*to_be_visualized.at[i,'patterns'], sep='; ')
            print(to_be_visualized.at[i,'table'].to_markdown())      
            print('Local entropy;',to_be_visualized.at[i,'entropy'])
            print()

    # Conditional entropy is simply the average of local entropies weighted by the frequency of the classes        
    freqs = pairs.applicable.value_counts(normalize=True).to_dict()
    return {'entropy':np.sum([freqs[k] * local_ents[k] for k in freqs]),'size':len(pairs)}   
 

The function `cont_ent_plus` allows one to condition entropy calculations on an extra feature such as the gender of nouns. That feature has to be made available as an extra column in the input dataframe.
The code is pretty straightforward: we group the data by values of the relevant feature, call `cond_ent`on each of the groups, and then compute a weighted average of the result.

In [8]:

def cond_ent_plus(cell1,cell2,feature,df,form_type,debug=False):
    '''Splits the data according to the information in one column, and returns conditional entropy 
    feature by feature'''
    local_ents = dict()
    freqs = dict()
    for i, g in df.groupby(feature):
        if debug:
            print('*'*50)
            print('Looking into feature', feature, 'value', i)
            print('*'*50)
            print()
            print()
        res = cond_ent(cell1,cell2,g,form_type,debug=debug)
        local_ents[i] = res['entropy']
        freqs[i] = res['size']
        if debug:
            print('Conditional entropy for', freqs[i] , 'lexemes with feature', i,':', local_ents[i])
    total = sum({freqs[k] for k in freqs})
    return {'entropy':np.sum([freqs[k] / total * local_ents[k] for k in freqs]),'size':total}   
